# Análisis exploratorio: Online Retail II

Exploración del dataset de ventas retail online (transacciones, productos, clientes y países).

**Contenido:**
- Carga y primera inspección
- Calidad de datos (nulos, duplicados, tipos)
- Estadísticas descriptivas
- Visualizaciones (distribuciones, series temporales, categorías)
- Conclusiones del EDA

**Dudas**  
¿cómo calcula los revenues?
tendría sentido quitar cuando tenemos identificada una compra?
varios modelos segun comportamiento del cliente?
entender bien como hace la construccion del data set agregado
 


In [ ]:
# Instalamos openpyxl si no lo tenemos (necesario para leer archivos .xlsx)
# Solo hay que ejecutar esta celda una vez
%pip install openpyxl -q

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', 20)
plt.style.use('ggplot')
sns.set_palette('husl')

## 1. Carga de datos

In [4]:

# Si el Excel tiene varias hojas, podemos cargar la primera o especificar por nombre
#df = pd.read_excel(DATA_PATH, sheet_name=1)

# Ruta al dataset
DATA_PATH = 'online_retail_II.xlsx'

# Leemos todas las hojas del Excel y las unimos en un solo DataFrame
xl = pd.ExcelFile(DATA_PATH)
print(f'Hojas encontradas: {xl.sheet_names}')

df = pd.concat(
    [pd.read_excel(DATA_PATH, sheet_name=s) for s in xl.sheet_names],
    ignore_index=True
)

print(f'Filas: {len(df):,} | Columnas: {len(df.columns)}')

Hojas encontradas: ['Year 2009-2010', 'Year 2010-2011']
Filas: 1,067,371 | Columnas: 8


## 2. Primera inspección

In [5]:

df.head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


In [9]:

df.dtypes

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
dtype: object

## 3. Calidad de datos

In [15]:
## chequear nulos y duplicados
# NULOS

nulos = df.isnull().sum()
porcentaje_nulos = (nulos / len(df) * 100).round(2)

pd.DataFrame({
    'Nulos': nulos,
    '% del total': porcentaje_nulos
}).query('Nulos > 0')

,Nulos,% del total
Description,4382,0.41
Customer ID,243007,22.77


In [16]:

# DUPLICADOS
n_duplicados = df.duplicated().sum()
print(f'Filas duplicadas: {n_duplicados:,} ({n_duplicados/len(df)*100:.2f}%)')

Filas duplicadas: 34,335 (3.22%)


In [17]:

# CANCELACIONES

# Las facturas que empiezan por "C" son cancelaciones.
# Las quitaremos porque no son compras reales.

# Creamos una columna auxiliar que indica si la factura es una cancelación
df['is_cancelled'] = df['Invoice'].astype(str).str.startswith('C')

n_cancelaciones = df['is_cancelled'].sum()
print(f'Transacciones canceladas: {n_cancelaciones:,} ({n_cancelaciones/len(df)*100:.2f}%)')
print(f'Transacciones normales:   {(~df["is_cancelled"]).sum():,}')

Transacciones canceladas: 19,494 (1.83%)
Transacciones normales:   1,047,877


In [19]:

# LIMPIEZA DEL DATASET

# aplico 4 filtros de limpieza:
# Eliminar duplicados
# Eliminar cancelaciones 
# Eliminar filas sin Customer ID
# Eliminar filas con Price <= 0 
#
# Guardamos el resultado en df_clean y mostramos cuántas filas quedan


print(f'Filas originales: {len(df):,}')

# 1. Quito duplicados
df_clean = df.drop_duplicates()
print(f'Tras quitar duplicados quedan: {len(df_clean):,}')

# 2. Quito cancelaciones
df_clean = df_clean[~df_clean['is_cancelled']].copy()
print(f'Tras quitar cancelaciones quedan: {len(df_clean):,}')

# 3. Quito filas sin Customer ID
df_clean = df_clean.dropna(subset=['Customer ID'])
print(f'Tras quitar sin Customer ID quedan: {len(df_clean):,}')

# 4. Quito filas con precio <= 0
df_clean = df_clean[df_clean['Price'] > 0].copy()
print(f'Tras quitar Price <= 0 quedan: {len(df_clean):,}')

# Elimino la columna auxiliar que ya no necesitamos
df_clean = df_clean.drop(columns=['is_cancelled'])

# Convierto Customer ID a entero (era float por los nulos)
df_clean['Customer ID'] = df_clean['Customer ID'].astype(int)

print(f'\n>>> Dataset limpio: {len(df_clean):,} filas ({len(df_clean)/len(df)*100:.1f}% del original)')

Filas originales: 1,067,371
Tras quitar duplicados quedan: 1,033,036
Tras quitar cancelaciones quedan: 1,013,932
Tras quitar sin Customer ID quedan: 779,495
Tras quitar Price <= 0 quedan: 779,425

>>> Dataset limpio: 779,425 filas (73.0% del original)


## 4. Estadísticas descriptivas

In [ ]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,1.067371e+06,1067371,1.067371e+06,824364.000000
mean,9.938898e+00,2011-01-02 21:13:55.394028544,4.649388e+00,15324.638504
min,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000
25%,1.000000e+00,2010-07-09 09:46:00,1.250000e+00,13975.000000
50%,3.000000e+00,2010-12-07 15:28:00,2.100000e+00,15255.000000
75%,1.000000e+01,2011-07-22 10:23:00,4.150000e+00,16797.000000
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000
std,1.727058e+02,NaN,1.235531e+02,1697.464450


In [24]:
# Creamos la columna Revenue (ingreso por línea de venta)
df_clean['Revenue'] = df_clean['Quantity'] * df_clean['Price']
df_clean.head(3)

# Estadísticas descriptivas sobre los datos limpios
print("=== Distribución de variables numéricas ===\n")
display(df_clean[['Quantity', 'Price', 'Revenue']].describe())

print("\n=== Dimensiones clave del negocio ===\n")
print(f"Clientes únicos:   {df_clean['Customer ID'].nunique():,}")
print(f"Facturas únicas:   {df_clean['Invoice'].nunique():,}")
print(f"Productos únicos:  {df_clean['StockCode'].nunique():,}")
print(f"Países:            {df_clean['Country'].nunique()}")
print(f"Rango temporal:    {df_clean['InvoiceDate'].min()} → {df_clean['InvoiceDate'].max()}")
print(f"Facturación total: {df_clean['Revenue'].sum():,.2f} €")

print("\n=== Estadísticas agregadas por cliente ===\n")
customer_stats = df_clean.groupby('Customer ID').agg(
    n_compras    = ('Invoice', 'nunique'),
    total_gasto  = ('Revenue', 'sum'),
    ticket_medio = ('Revenue', 'mean'),
    items_total  = ('Quantity', 'sum')
)
display(customer_stats.describe())

print("\n=== Top 10 países por facturación ===\n")
display(df_clean.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10))

print("\n=== Percentiles extremos (detección de outliers) ===\n")
display(df_clean[['Quantity', 'Price', 'Revenue']].quantile([0.01, 0.05, 0.95, 0.99]))

=== Distribución de variables numéricas ===



,Quantity,Price,Revenue
count,779425.000000,779425.000000,779425.000000
mean,13.489370,3.218488,22.291823
std,145.855814,29.676140,227.427075
min,1.000000,0.001000,0.001000
25%,2.000000,1.250000,4.950000
50%,6.000000,1.950000,12.480000
75%,12.000000,3.750000,19.800000
max,80995.000000,10953.500000,168469.600000



=== Dimensiones clave del negocio ===

Clientes únicos:   5,878
Facturas únicas:   36,969
Productos únicos:  4,631
Países:            41
Rango temporal:    2009-12-01 07:45:00 → 2011-12-09 12:50:00
Facturación total: 17,374,804.27 €

=== Estadísticas agregadas por cliente ===



,n_compras,total_gasto,ticket_medio,items_total
count,5878.000000,5878.000000,5878.000000,5878.000000
mean,6.289384,2955.904095,48.301822,1788.695475
std,13.009406,14440.852688,780.176769,8876.297196
min,1.000000,2.950000,2.135778,1.000000
25%,1.000000,342.280000,11.564180,187.000000
50%,3.000000,867.740000,17.367639,480.000000
75%,7.000000,2248.305000,24.181900,1350.000000
max,398.000000,580987.040000,56157.500000,367193.000000



=== Top 10 países por facturación ===



Country
United Kingdom    1.438923e+07
EIRE              6.165705e+05
Netherlands       5.540381e+05
Germany           4.250197e+05
France            3.487690e+05
Australia         1.692835e+05
Spain             1.083325e+05
Switzerland       1.000619e+05
Sweden            9.151582e+04
Denmark           6.858069e+04
Name: Revenue, dtype: float64


=== Percentiles extremos (detección de outliers) ===



,Quantity,Price,Revenue
0.01,1.0,0.29,0.60
0.05,1.0,0.42,1.25
0.95,36.0,8.50,67.50
0.99,144.0,14.95,203.52


## 5. Visualizaciones

## 6. Agrupación por cliente (Customer ID)

Por cada cliente: **suma de tickets en el año** (facturación total), **número de compras** (facturas distintas) y **ticket medio** por compra.

## 7. Modelo: probabilidad de recompra

Modelo supervisado para estimar la probabilidad de que un cliente **vuelva a comprar** después de una fecha de corte.

Estrategia:
- Definimos una **fecha de corte** (`cutoff_date`).
- Con los datos **antes** de esa fecha construimos variables por cliente (RFM simplificado: recencia, frecuencia, gasto total, ticket medio, país, etc.).
- Con los datos **después** de la fecha etiquetamos si el cliente **recompró (1)** o **no recompró (0)**.
- Entrenamos una **regresión logística** para predecir esa probabilidad de recompra.

### (Opcional) Búsqueda de hiperparámetros
randomsearch, gridsearch, optuna, 


## 8. Modelo: explicabilidad
indicar las variables más relevantes del modelo. COn Dalex o más sencillo, los propios  tiene manera de obtener las variables:   
Ej. 

from xgboost import plot_importance
import matplotlib.pyplot as plt

plot_importance(model)
plt.show()

